# Chapter 6: Fuchsia City & Cinnabar Island -- Instrumental Variables & Regression Discontinuity

---

This chapter is a two-part adventure:

**Part I -- Fuchsia City & the Safari Zone Lottery (Instrumental Variables)**  
Fuchsia City sits at the southern edge of Kanto, home to Koga's Poison-type Gym and the famous
Safari Zone. Access to the Safari Zone is limited, but the city runs a *lottery* to award entry
tickets. This random lottery is our **instrument** -- it shifts who attends the Safari Zone
without directly affecting battle outcomes. We will use it to estimate the causal effect of
Safari Zone attendance on post-visit battle performance.

**Part II -- Cinnabar Island & Happiness Evolution (Regression Discontinuity)**  
On Cinnabar Island, Blaine's Volcano Gym is not the only source of fire. Pokemon evolve through
happiness -- and there is a sharp threshold at 220 happiness points. This natural cutoff
gives us a **regression discontinuity design** to measure the causal effect of evolution on
battle performance.

By the end, you will have earned both the **Soul Badge** and the **Volcano Badge**.

---
# Part I: Instrumental Variables (Fuchsia City)

---
## 6.1 Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Ensure kanto_utils is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from kanto_utils import (
    load_safari_lottery, load_happiness, apply_kanto_theme,
    wald_estimator, two_stage_ls, sharp_rdd, rdd_plot,
    balance_table,
    oak_says, blue_says, blues_mistake, badge_earned,
)

apply_kanto_theme()
np.random.seed(151)

# Load datasets
safari = load_safari_lottery()
happiness = load_happiness()

print(f"Safari Zone lottery: {len(safari)} trainers, {safari.shape[1]} variables")
print(f"Happiness evolution: {len(happiness)} Pokemon, {happiness.shape[1]} variables")
print()
print("Safari columns:", list(safari.columns))
print("Happiness columns:", list(happiness.columns))

In [ ]:
oak_says(
    "Welcome to the southern coast of Kanto! In <b>Fuchsia City</b>, the Safari Zone "
    "runs a lottery to decide who gets in. This randomisation is exactly what we need "
    "for an <b>instrumental variables</b> strategy. Then we will sail to <b>Cinnabar Island</b>, "
    "where a happiness threshold creates a natural experiment perfect for "
    "<b>regression discontinuity</b>."
)

---
## 6.2 The Endogeneity Problem

In [ ]:
# Naive OLS: battle_wins_post ~ safari_attended
naive_iv = smf.ols('battle_wins_post ~ safari_attended', data=safari).fit()
print("=== NAIVE OLS: battle_wins_post ~ safari_attended ===")
print(naive_iv.summary().tables[1])
print(f"\nNaive effect of Safari attendance: {naive_iv.params['safari_attended']:.4f}")

In [ ]:
# But safari_attended is correlated with patience (an unobserved confounder)!
# Reveal the hidden confounder.
print("=== Revealing the hidden confounder: patience ===")
print(f"Correlation(safari_attended, patience): {safari['safari_attended'].corr(safari['patience']):.4f}")
print(f"Correlation(patience, battle_wins_post): {safari['patience'].corr(safari['battle_wins_post']):.4f}")

# OLS with patience controlled
controlled_iv = smf.ols('battle_wins_post ~ safari_attended + patience', data=safari).fit()
print(f"\nNaive estimate (no patience):     {naive_iv.params['safari_attended']:.4f}")
print(f"Controlled estimate (+ patience):  {controlled_iv.params['safari_attended']:.4f}")
print(f"Confounding bias:                  {naive_iv.params['safari_attended'] - controlled_iv.params['safari_attended']:.4f}")

In [ ]:
# Visualise the confounding
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: patience by safari attendance
for attended, color, label in [(0, '#3B4CCA', 'Did not attend'), (1, '#EE1515', 'Attended')]:
    subset = safari[safari['safari_attended'] == attended]
    ax1.hist(subset['patience'], bins=25, alpha=0.6, color=color, label=label, density=True)
ax1.set_xlabel('Patience')
ax1.set_ylabel('Density')
ax1.set_title('Patience by Safari Attendance')
ax1.legend()

# Right: patience vs battle_wins_post
ax2.scatter(safari['patience'], safari['battle_wins_post'], s=15, alpha=0.4, color='#3B4CCA')
z = np.polyfit(safari['patience'], safari['battle_wins_post'], 1)
xs = np.linspace(safari['patience'].min(), safari['patience'].max(), 100)
ax2.plot(xs, np.polyval(z, xs), color='#EE1515', lw=2, label=f'slope = {z[0]:.3f}')
ax2.set_xlabel('Patience')
ax2.set_ylabel('Battle Wins (Post)')
ax2.set_title('Patience Confounds the Outcome')
ax2.legend()

fig.tight_layout()
plt.show()

In [ ]:
blues_mistake(
    "Trainers who went to the Safari Zone won way more battles afterwards. "
    "The Safari Zone CLEARLY makes you a better trainer!",
    "Trainers who attend the Safari Zone tend to be more patient, and patience "
    "independently predicts battle wins. The naive OLS estimate conflates the "
    "causal effect of attendance with the confounding effect of patience. "
    "We need an instrument to break this endogeneity."
)

---
## 6.3 The Instrument: Safari Zone Lottery

In [ ]:
# Check the three IV conditions for lottery_won

# (a) RANDOM ASSIGNMENT: balanced covariates across lottery winners/losers
covariates = ['team_level', 'trainer_experience', 'badges', 'strategy_score',
              'battle_wins_pre', 'patience', 'motivation']

bal = balance_table(safari, 'lottery_won', covariates)
print("=== Covariate Balance: Lottery Winners vs Losers ===")
display(bal[['mean_treated', 'mean_control', 'std_diff', 'p_value']])

print(f"\nAll p-values > 0.05: {(bal['p_value'] > 0.05).all()}")
print(f"Max |std_diff|: {bal['std_diff'].abs().max():.4f}")

In [ ]:
# (b) RELEVANCE: lottery_won predicts safari_attended (strong first stage)
first_stage = smf.ols('safari_attended ~ lottery_won', data=safari).fit()
print("=== First Stage: safari_attended ~ lottery_won ===")
print(first_stage.summary().tables[1])
print(f"\nFirst-stage coefficient: {first_stage.params['lottery_won']:.4f}")
print(f"First-stage F-statistic: {first_stage.fvalue:.2f}")
print(f"Rule of thumb: F > 10 is strong. {'STRONG' if first_stage.fvalue > 10 else 'WEAK'} instrument.")

In [ ]:
# (c) EXOGENEITY: lottery_won is uncorrelated with patience
corr_z_patience = safari['lottery_won'].corr(safari['patience'])
_, p_z_patience = stats.ttest_ind(
    safari.loc[safari['lottery_won']==1, 'patience'],
    safari.loc[safari['lottery_won']==0, 'patience']
)

print(f"Correlation(lottery_won, patience): {corr_z_patience:.4f}")
print(f"T-test p-value: {p_z_patience:.4f}")
print(f"Exclusion restriction appears satisfied: {'YES' if p_z_patience > 0.05 else 'NO'}")

In [ ]:
oak_says(
    "A valid instrument must satisfy three conditions: "
    "<b>(1) Relevance</b> -- the instrument must predict the treatment (F > 10); "
    "<b>(2) Independence / Random Assignment</b> -- the instrument is uncorrelated with confounders; "
    "<b>(3) Exclusion Restriction</b> -- the instrument affects the outcome ONLY through the treatment. "
    "The lottery satisfies all three: it is random, strongly predicts attendance, "
    "and winning a lottery ticket should not directly make you a better battler."
)

---
## 6.4 Manual Two-Stage Least Squares (2SLS)

In [ ]:
# --- Step-by-step 2SLS ---

# STAGE 1: Regress endogenous treatment on instrument
stage1 = smf.ols('safari_attended ~ lottery_won', data=safari).fit()
safari['safari_hat'] = stage1.fittedvalues

print("=== STAGE 1: safari_attended ~ lottery_won ===")
print(f"  Coefficient: {stage1.params['lottery_won']:.4f}")
print(f"  R-squared:   {stage1.rsquared:.4f}")
print(f"  F-stat:      {stage1.fvalue:.2f}")

# STAGE 2: Regress outcome on predicted treatment
stage2 = smf.ols('battle_wins_post ~ safari_hat', data=safari).fit()

print(f"\n=== STAGE 2: battle_wins_post ~ safari_hat ===")
print(f"  2SLS coefficient: {stage2.params['safari_hat']:.4f}")

# Compare with Wald estimator from kanto_utils
wald = wald_estimator(
    safari['battle_wins_post'].values,
    safari['safari_attended'].values,
    safari['lottery_won'].values
)

print(f"\n=== Wald Estimator (kanto_utils) ===")
print(f"  LATE estimate:  {wald['estimate']:.4f}")
print(f"  SE:             {wald['se']:.4f}")
print(f"  95% CI:         [{wald['ci_lower']:.4f}, {wald['ci_upper']:.4f}]")
print(f"  First stage:    {wald['first_stage']:.4f}")
print(f"  Reduced form:   {wald['reduced_form']:.4f}")

print(f"\n=== Comparison ===")
print(f"  Manual 2SLS:    {stage2.params['safari_hat']:.4f}")
print(f"  Wald estimator: {wald['estimate']:.4f}")
print(f"  Naive OLS:      {naive_iv.params['safari_attended']:.4f}")

In [ ]:
# Also verify with two_stage_ls
tsls = two_stage_ls(
    safari['battle_wins_post'].values,
    safari['safari_attended'].values,
    safari['lottery_won'].values
)
print(f"two_stage_ls estimate: {tsls['estimate']:.4f}")
print(f"two_stage_ls SE:       {tsls['se']:.4f}")
print(f"two_stage_ls F-stat:   {tsls['first_stage_f']:.2f}")

---
## 6.5 Complier Analysis

In [ ]:
# The dataset includes compliance_type -- let's examine the types
comp_counts = safari['compliance_type'].value_counts()
print("=== Compliance Type Distribution ===")
print(comp_counts)
print(f"\nTotal: {len(safari)}")
print(f"Complier share: {comp_counts.get('complier', 0) / len(safari):.3f}")

In [ ]:
# Visualise compliance types
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: compliance type bar chart
comp_colors = {'complier': '#4DAD5B', 'always_taker': '#EE1515', 'never_taker': '#3B4CCA'}
comp_labels = comp_counts.index.tolist()
ax1.bar(comp_labels, comp_counts.values,
        color=[comp_colors.get(c, '#888') for c in comp_labels],
        edgecolor='white')
ax1.set_ylabel('Count')
ax1.set_title('Compliance Type Distribution')
for i, (label, count) in enumerate(zip(comp_labels, comp_counts.values)):
    ax1.text(i, count + 5, f'{count}\n({count/len(safari)*100:.1f}%)',
             ha='center', fontweight='bold')

# Right: treatment effect by compliance type
for ctype, color in comp_colors.items():
    subset = safari[safari['compliance_type'] == ctype]
    if len(subset) > 0:
        effect = subset['battle_wins_post'].mean() - subset['battle_wins_pre'].mean()
        ax2.bar(ctype, effect, color=color, edgecolor='white')
        ax2.text(list(comp_colors.keys()).index(ctype), effect + 0.3,
                 f'{effect:.2f}', ha='center', fontweight='bold')

ax2.set_ylabel('Mean (post - pre) battle wins')
ax2.set_title('Pre-Post Change by Compliance Type')
ax2.axhline(0, color='gray', ls=':', lw=1)

fig.tight_layout()
plt.show()

In [ ]:
# Compute LATE directly among compliers vs ATE across all
compliers = safari[safari['compliance_type'] == 'complier']

# Among compliers: those who won lottery attended, those who lost did not
late_direct = (
    compliers.loc[compliers['lottery_won']==1, 'battle_wins_post'].mean() -
    compliers.loc[compliers['lottery_won']==0, 'battle_wins_post'].mean()
)

# ATE across everyone (using all types)
ate_all = (
    safari.loc[safari['safari_attended']==1, 'battle_wins_post'].mean() -
    safari.loc[safari['safari_attended']==0, 'battle_wins_post'].mean()
)

print(f"LATE (IV/Wald estimate):     {wald['estimate']:.4f}")
print(f"LATE (direct, compliers):    {late_direct:.4f}")
print(f"Naive ATE (all attendees):   {ate_all:.4f}")
print(f"\nLATE != ATE: the IV estimates the effect for COMPLIERS only,")
print(f"not for always-takers or never-takers.")

In [ ]:
oak_says(
    "The IV/Wald estimator recovers the <b>Local Average Treatment Effect (LATE)</b>: "
    "the causal effect specifically for <b>compliers</b> -- trainers whose Safari Zone "
    "attendance was actually changed by the lottery. "
    "The formal taxonomy of compliance types: "
    "<b>Compliers</b> attend if and only if they win the lottery. "
    "<b>Always-takers</b> attend regardless (they find another way in). "
    "<b>Never-takers</b> never attend even if they win. "
    "Under the monotonicity assumption (no defiers), IV = LATE."
)

---
# Part II: Regression Discontinuity (Cinnabar Island)

---
## 6.6 Sharp RDD Setup

In [ ]:
# The happiness threshold for evolution is 220
CUTOFF = 220

print(f"Happiness evolution dataset: {len(happiness)} Pokemon")
print(f"Cutoff: {CUTOFF}")
print(f"Above threshold: {happiness['above_threshold'].sum()} ({happiness['above_threshold'].mean()*100:.1f}%)")
print(f"Evolved: {happiness['evolved'].sum()} ({happiness['evolved'].mean()*100:.1f}%)")

# Scatter plot with the discontinuity
fig, ax = plt.subplots(figsize=(10, 6))

below = happiness[happiness['happiness_score'] < CUTOFF]
above = happiness[happiness['happiness_score'] >= CUTOFF]

ax.scatter(below['happiness_score'], below['battle_performance_post'],
           s=15, alpha=0.4, color='#3B4CCA', label='Below threshold')
ax.scatter(above['happiness_score'], above['battle_performance_post'],
           s=15, alpha=0.4, color='#EE1515', label='Above threshold')
ax.axvline(CUTOFF, color='#FFD733', ls='--', lw=2.5, label=f'Cutoff = {CUTOFF}')
ax.set_xlabel('Happiness Score')
ax.set_ylabel('Battle Performance (Post)')
ax.set_title('Happiness-Evolution RDD: Raw Data')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Use the rdd_plot utility for a polished version
fig, ax = plt.subplots(figsize=(10, 6))
rdd_plot(
    happiness['happiness_score'].values,
    happiness['battle_performance_post'].values,
    cutoff=CUTOFF,
    bandwidth=40,
    n_bins=30,
    ax=ax
)
ax.set_xlabel('Happiness Score')
ax.set_ylabel('Battle Performance (Post)')
ax.set_title('RDD Plot: Happiness -> Battle Performance')
plt.tight_layout()
plt.show()

---
## 6.7 Local Linear Estimation

In [ ]:
# Estimate the RDD effect using sharp_rdd with different bandwidths
running = happiness['happiness_score'].values
outcome_rdd = happiness['battle_performance_post'].values

# Default bandwidth (Silverman ROT)
rdd_default = sharp_rdd(running, outcome_rdd, CUTOFF)
print("=== Sharp RDD (default bandwidth) ===")
print(f"  Estimate:  {rdd_default['estimate']:.4f}")
print(f"  SE:        {rdd_default['se']:.4f}")
print(f"  95% CI:    [{rdd_default['ci_lower']:.4f}, {rdd_default['ci_upper']:.4f}]")
print(f"  Bandwidth: {rdd_default['bandwidth']:.2f}")
print(f"  N left:    {rdd_default['n_left']}")
print(f"  N right:   {rdd_default['n_right']}")

# Compare specific bandwidths
print("\n=== Bandwidth comparison ===")
for bw in [15, 25, 35, 50]:
    try:
        res = sharp_rdd(running, outcome_rdd, CUTOFF, bandwidth=bw)
        print(f"  bw={bw:3d}: estimate={res['estimate']:7.3f}, "
              f"SE={res['se']:.3f}, "
              f"CI=[{res['ci_lower']:.3f}, {res['ci_upper']:.3f}], "
              f"N={res['n_left']+res['n_right']}")
    except ValueError as e:
        print(f"  bw={bw:3d}: {e}")

In [ ]:
# Visualise the local linear fits on each side
bw_plot = 35

fig, ax = plt.subplots(figsize=(10, 6))
mask = np.abs(running - CUTOFF) <= bw_plot

below_m = mask & (running < CUTOFF)
above_m = mask & (running >= CUTOFF)

ax.scatter(running[below_m], outcome_rdd[below_m], s=20, alpha=0.4, color='#3B4CCA')
ax.scatter(running[above_m], outcome_rdd[above_m], s=20, alpha=0.4, color='#EE1515')

# Fit and plot local linear regressions
for m, color in [(below_m, '#3B4CCA'), (above_m, '#EE1515')]:
    if m.sum() >= 2:
        coeffs = np.polyfit(running[m], outcome_rdd[m], 1)
        xs = np.linspace(running[m].min(), running[m].max(), 200)
        ax.plot(xs, np.polyval(coeffs, xs), color=color, lw=2.5)

# Mark the estimated jump
res = sharp_rdd(running, outcome_rdd, CUTOFF, bandwidth=bw_plot)
ax.axvline(CUTOFF, color='#FFD733', ls='--', lw=2.5)
ax.annotate(f'Jump = {res["estimate"]:.2f}',
            xy=(CUTOFF, np.mean(outcome_rdd[mask])),
            xytext=(CUTOFF + 10, np.mean(outcome_rdd[mask]) + 5),
            fontsize=12, fontweight='bold', color='#EE1515',
            arrowprops=dict(arrowstyle='->', color='#EE1515', lw=1.5))

ax.set_xlabel('Happiness Score')
ax.set_ylabel('Battle Performance (Post)')
ax.set_title(f'Local Linear RDD (bandwidth = {bw_plot})')
plt.tight_layout()
plt.show()

---
## 6.8 Bandwidth Sensitivity

In [ ]:
# Manual bandwidth sensitivity analysis
bandwidths = np.arange(8, 55, 2)
bw_results = []

for bw in bandwidths:
    try:
        res = sharp_rdd(running, outcome_rdd, CUTOFF, bandwidth=bw)
        bw_results.append({
            'bandwidth': bw,
            'estimate': res['estimate'],
            'se': res['se'],
            'ci_lower': res['ci_lower'],
            'ci_upper': res['ci_upper'],
            'n_total': res['n_left'] + res['n_right']
        })
    except (ValueError, np.linalg.LinAlgError):
        pass

bw_df = pd.DataFrame(bw_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: estimate vs bandwidth
ax1.plot(bw_df['bandwidth'], bw_df['estimate'], 'o-', color='#EE1515', lw=2, markersize=5)
ax1.fill_between(bw_df['bandwidth'], bw_df['ci_lower'], bw_df['ci_upper'],
                 color='#EE1515', alpha=0.15)
ax1.axhline(0, color='gray', ls=':', lw=1)
ax1.axvline(rdd_default['bandwidth'], color='#FFD733', ls='--', lw=1.5,
            label=f'Default bw = {rdd_default["bandwidth"]:.1f}')
ax1.set_xlabel('Bandwidth')
ax1.set_ylabel('RDD Estimate')
ax1.set_title('RDD Estimate vs Bandwidth')
ax1.legend(fontsize=9)

# Right: sample size vs bandwidth
ax2.plot(bw_df['bandwidth'], bw_df['n_total'], 's-', color='#3B4CCA', lw=2, markersize=5)
ax2.set_xlabel('Bandwidth')
ax2.set_ylabel('Observations Used')
ax2.set_title('Sample Size vs Bandwidth')

fig.tight_layout()
plt.show()

---
## 6.9 McCrary Density Test

In [ ]:
# McCrary test: check for manipulation (bunching) at the cutoff
# If trainers can manipulate their Pokemon's happiness to game the threshold,
# we would see a suspicious density jump at the cutoff.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: full histogram
ax1.hist(happiness['happiness_score'], bins=50, color='#3B4CCA', alpha=0.7, edgecolor='white')
ax1.axvline(CUTOFF, color='#FFD733', ls='--', lw=2.5, label=f'Cutoff = {CUTOFF}')
ax1.set_xlabel('Happiness Score')
ax1.set_ylabel('Count')
ax1.set_title('Full Distribution of Running Variable')
ax1.legend()

# Right: zoomed in around cutoff
near_cutoff = happiness[(happiness['happiness_score'] >= CUTOFF - 30) &
                        (happiness['happiness_score'] <= CUTOFF + 30)]
bins = np.arange(CUTOFF - 30, CUTOFF + 31, 3)
ax2.hist(near_cutoff['happiness_score'], bins=bins, color='#3B4CCA', alpha=0.7, edgecolor='white')
ax2.axvline(CUTOFF, color='#FFD733', ls='--', lw=2.5, label=f'Cutoff = {CUTOFF}')
ax2.set_xlabel('Happiness Score')
ax2.set_ylabel('Count')
ax2.set_title('Density Near Cutoff (McCrary Check)')
ax2.legend()

fig.tight_layout()
plt.show()

# Simple density test: compare counts in bins just left and right of cutoff
bin_width = 5
n_left_bin = ((happiness['happiness_score'] >= CUTOFF - bin_width) &
              (happiness['happiness_score'] < CUTOFF)).sum()
n_right_bin = ((happiness['happiness_score'] >= CUTOFF) &
               (happiness['happiness_score'] < CUTOFF + bin_width)).sum()

print(f"Count in [{CUTOFF-bin_width}, {CUTOFF}): {n_left_bin}")
print(f"Count in [{CUTOFF}, {CUTOFF+bin_width}): {n_right_bin}")
print(f"Ratio (right/left): {n_right_bin / max(n_left_bin, 1):.3f}")
print(f"No obvious bunching: {'YES' if 0.6 < n_right_bin/max(n_left_bin,1) < 1.5 else 'SUSPICIOUS'}")

---
## 6.10 Placebo Tests & Fuzzy RDD

In [ ]:
# --- (a) Placebo tests: pre-treatment covariates should NOT show discontinuities ---
placebo_vars = ['pokemon_level', 'friendship_days', 'trainer_skill']

print("=== Placebo RDD Tests (pre-treatment covariates at cutoff) ===")
placebo_results = []
for var in placebo_vars:
    try:
        res = sharp_rdd(running, happiness[var].values, CUTOFF,
                        bandwidth=rdd_default['bandwidth'])
        placebo_results.append({
            'variable': var,
            'estimate': res['estimate'],
            'se': res['se'],
            'p_value': 2 * stats.norm.sf(abs(res['estimate'] / res['se'])),
        })
        print(f"  {var:25s}: effect = {res['estimate']:7.3f}, "
              f"SE = {res['se']:.3f}, "
              f"p = {placebo_results[-1]['p_value']:.4f}")
    except (ValueError, np.linalg.LinAlgError) as e:
        print(f"  {var:25s}: {e}")

placebo_df = pd.DataFrame(placebo_results)
print(f"\nAll placebo p-values > 0.05: {(placebo_df['p_value'] > 0.05).all() if len(placebo_df) > 0 else 'N/A'}")

In [ ]:
# Visualise placebo tests
n_vars = len(placebo_vars)
fig, axes = plt.subplots(1, n_vars, figsize=(5*n_vars, 4.5))
if n_vars == 1:
    axes = [axes]

for ax, var in zip(axes, placebo_vars):
    rdd_plot(
        running, happiness[var].values,
        cutoff=CUTOFF, bandwidth=rdd_default['bandwidth'],
        n_bins=20, ax=ax
    )
    ax.set_title(f'Placebo: {var}')
    ax.set_xlabel('Happiness Score')
    ax.set_ylabel(var)

fig.suptitle('Placebo Tests: Pre-Treatment Covariates Should Show No Jump',
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- (b) Fuzzy RDD ---
# trainer_pressed_b creates a gap between eligibility and actual evolution.
# Not all Pokemon above the threshold actually evolve (some trainers press B!).

print("=== Fuzzy RDD: Eligibility vs Actual Evolution ===")
print(f"Above threshold (eligible): {happiness['above_threshold'].sum()}")
print(f"Actually evolved:           {happiness['evolved'].sum()}")
print(f"Trainer pressed B:          {happiness['trainer_pressed_b'].sum()}")

# Cross-tabulation
print("\nCross-tab (above_threshold x evolved):")
ct = pd.crosstab(happiness['above_threshold'], happiness['evolved'],
                 margins=True, margins_name='Total')
display(ct)

# First stage: above_threshold -> evolved
first_stage_rdd = smf.ols('evolved ~ above_threshold', data=happiness).fit()
print(f"\nFirst stage (eligibility -> evolution): {first_stage_rdd.params['above_threshold']:.4f}")

In [ ]:
# Fuzzy RDD = Sharp RDD (reduced form) / First stage at cutoff
# Reduced form: effect of being above threshold on battle_performance_post
reduced_form_rdd = sharp_rdd(running, outcome_rdd, CUTOFF, bandwidth=rdd_default['bandwidth'])

# First stage at cutoff: jump in P(evolved) at the threshold
first_stage_at_cutoff = sharp_rdd(
    running, happiness['evolved'].values, CUTOFF,
    bandwidth=rdd_default['bandwidth']
)

fuzzy_estimate = reduced_form_rdd['estimate'] / first_stage_at_cutoff['estimate']

print("=== Fuzzy RDD Estimation ===")
print(f"  Reduced form (ITT): {reduced_form_rdd['estimate']:.4f}")
print(f"  First stage jump:   {first_stage_at_cutoff['estimate']:.4f}")
print(f"  Fuzzy RDD (LATE):   {fuzzy_estimate:.4f}")

# Compare with sharp RDD (which assumes perfect compliance)
print(f"\n  Sharp RDD estimate: {reduced_form_rdd['estimate']:.4f}")
print(f"  Fuzzy RDD estimate: {fuzzy_estimate:.4f}")
print(f"  The fuzzy estimate is larger because it accounts for")
print(f"  imperfect compliance (some trainers press B to cancel evolution).")

In [ ]:
# Visualise fuzzy RDD: show the jump in treatment take-up
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: evolution probability by happiness
rdd_plot(
    running, happiness['evolved'].values,
    cutoff=CUTOFF, bandwidth=40, n_bins=25, ax=ax1
)
ax1.set_ylabel('P(Evolved)')
ax1.set_title('First Stage: Evolution Probability')

# Right: outcome (battle performance)
rdd_plot(
    running, outcome_rdd,
    cutoff=CUTOFF, bandwidth=40, n_bins=25, ax=ax2
)
ax2.set_ylabel('Battle Performance')
ax2.set_title('Reduced Form: Battle Performance')

fig.suptitle('Fuzzy RDD: First Stage & Reduced Form', fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
oak_says(
    "In a <b>fuzzy RDD</b>, crossing the threshold does not guarantee treatment -- "
    "it only increases the probability. Here, some trainers press the B button to cancel "
    "evolution even after their Pokemon crosses the happiness threshold. "
    "The fuzzy RDD estimate is the ratio of the reduced-form jump (in the outcome) to the "
    "first-stage jump (in treatment take-up). Just like IV, it recovers a <b>LATE</b> for "
    "compliers at the threshold."
)

---
## Challenges

Complete these exercises to earn the **Soul Badge + Volcano Badge**!

### Challenge 1: Test a Different Instrument

Suppose we consider using `motivation` as an instrument for `safari_attended`.

**Task:** Check whether `motivation` satisfies the three IV conditions (relevance, independence,
exclusion). Does it pass? Why or why not?

In [ ]:
# Challenge 1: Test motivation as an instrument

# (a) Relevance: does motivation predict safari_attended?
rel_test = smf.ols('safari_attended ~ motivation', data=safari).fit()
print("=== Relevance Test ===")
print(f"  Coefficient: {rel_test.params['motivation']:.6f}")
print(f"  F-statistic: {rel_test.fvalue:.2f}")
print(f"  Strong instrument (F>10): {rel_test.fvalue > 10}")

# (b) Independence: is motivation correlated with the confounder (patience)?
corr_motiv_patience = safari['motivation'].corr(safari['patience'])
_, p_motiv_patience = stats.pearsonr(safari['motivation'], safari['patience'])
print(f"\n=== Independence Test ===")
print(f"  Corr(motivation, patience): {corr_motiv_patience:.4f}")
print(f"  P-value: {p_motiv_patience:.6f}")
print(f"  Independent of confounder: {p_motiv_patience > 0.05}")

# (c) Exclusion: does motivation directly affect battle_wins_post?
# Control for safari_attended -- residual effect would violate exclusion
excl_test = smf.ols('battle_wins_post ~ safari_attended + motivation', data=safari).fit()
print(f"\n=== Exclusion Restriction Check ===")
print(f"  Coefficient on motivation (controlling for attendance): {excl_test.params['motivation']:.4f}")
print(f"  P-value: {excl_test.pvalues['motivation']:.4f}")

print("\n=== Verdict ===")
print("motivation is likely NOT a valid instrument because:")
if corr_motiv_patience < -0.1 or corr_motiv_patience > 0.1:
    print("  - It may be correlated with the confounder (patience)")
if excl_test.pvalues['motivation'] < 0.05:
    print("  - It has a direct effect on the outcome (violates exclusion)")
if rel_test.fvalue <= 10:
    print("  - It may be a weak instrument (F <= 10)")

### Challenge 2: RDD with Different Kernel Functions

**Task:** Estimate the RDD effect using triangular, uniform, and Epanechnikov kernels.
Compare the estimates and SEs. Which kernel gives the narrowest confidence interval?

In [ ]:
# Challenge 2: Compare kernel functions
kernels = ['triangular', 'uniform', 'epanechnikov']
bw_test = rdd_default['bandwidth']  # use the default bandwidth for fair comparison

kernel_results = []
for kern in kernels:
    res = sharp_rdd(running, outcome_rdd, CUTOFF, bandwidth=bw_test, kernel=kern)
    kernel_results.append({
        'kernel': kern,
        'estimate': res['estimate'],
        'se': res['se'],
        'ci_lower': res['ci_lower'],
        'ci_upper': res['ci_upper'],
        'ci_width': res['ci_upper'] - res['ci_lower'],
    })

kernel_df = pd.DataFrame(kernel_results)
display(kernel_df.round(4))

# Visualise
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#EE1515', '#3B4CCA', '#4DAD5B']
for i, (_, row) in enumerate(kernel_df.iterrows()):
    ax.errorbar(i, row['estimate'], yerr=1.96*row['se'],
                fmt='o', color=colors[i], capsize=8, markersize=10, lw=2)

ax.set_xticks(range(len(kernels)))
ax.set_xticklabels(kernels)
ax.set_ylabel('RDD Estimate')
ax.set_title(f'RDD Estimates by Kernel (bw = {bw_test:.1f})')
ax.axhline(0, color='gray', ls=':', lw=1)
plt.tight_layout()
plt.show()

narrowest = kernel_df.loc[kernel_df['ci_width'].idxmin(), 'kernel']
print(f"Narrowest CI: {narrowest} kernel")

### Challenge 3: Donut-Hole RDD

In a donut-hole RDD, we **exclude** observations very close to the cutoff
(where manipulation might be most likely) and re-estimate.

**Task:** Exclude all Pokemon within 5 happiness points of the cutoff (i.e., happiness in
[215, 225]) and re-estimate the sharp RDD. Compare with the full-sample estimate.

In [ ]:
# Challenge 3: Donut-hole RDD
donut_radius = 5
donut_mask = np.abs(running - CUTOFF) > donut_radius

running_donut = running[donut_mask]
outcome_donut = outcome_rdd[donut_mask]

print(f"Observations removed (within {donut_radius} of cutoff): {(~donut_mask).sum()}")
print(f"Remaining observations: {donut_mask.sum()}")

# Re-estimate
rdd_donut = sharp_rdd(running_donut, outcome_donut, CUTOFF, bandwidth=rdd_default['bandwidth'])

print(f"\n=== Full Sample RDD ===")
print(f"  Estimate: {rdd_default['estimate']:.4f}, SE: {rdd_default['se']:.4f}")

print(f"\n=== Donut-Hole RDD (exclude within {donut_radius} of cutoff) ===")
print(f"  Estimate: {rdd_donut['estimate']:.4f}, SE: {rdd_donut['se']:.4f}")

print(f"\nDifference: {rdd_donut['estimate'] - rdd_default['estimate']:.4f}")
print("If the donut-hole estimate is similar, the result is robust to potential manipulation near the cutoff.")

# Visualise
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

rdd_plot(running, outcome_rdd, CUTOFF, bandwidth=rdd_default['bandwidth'], ax=ax1)
ax1.set_title(f'Full Sample (est = {rdd_default["estimate"]:.3f})')

rdd_plot(running_donut, outcome_donut, CUTOFF, bandwidth=rdd_default['bandwidth'], ax=ax2)
ax2.set_title(f'Donut Hole, r={donut_radius} (est = {rdd_donut["estimate"]:.3f})')

fig.suptitle('Donut-Hole Robustness Check', fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

### Challenge 4: LATE as a Function of First-Stage Strength

The IV estimate can be decomposed as LATE = Reduced Form / First Stage.  
What happens to the LATE when the first stage gets weaker?

**Task:** Simulate weaker instruments by adding noise to `lottery_won`. For noise levels
from 0 to 0.5, randomly flip that fraction of lottery assignments and re-estimate the Wald
estimate. Plot LATE vs first-stage coefficient.

In [ ]:
# Challenge 4: LATE vs first-stage strength
rng = np.random.default_rng(151)
noise_levels = np.arange(0, 0.51, 0.02)
late_sim = []

z_orig = safari['lottery_won'].values.copy()
y_iv = safari['battle_wins_post'].values
d_iv = safari['safari_attended'].values

for noise in noise_levels:
    # Randomly flip 'noise' fraction of the instrument
    z_noisy = z_orig.copy()
    flip_mask = rng.random(len(z_noisy)) < noise
    z_noisy[flip_mask] = 1 - z_noisy[flip_mask]
    
    try:
        wald_res = wald_estimator(y_iv, d_iv, z_noisy)
        # First-stage strength
        fs = wald_res['first_stage']
        late_sim.append({
            'noise_level': noise,
            'first_stage': fs,
            'late': wald_res['estimate'],
            'se': wald_res['se'],
        })
    except ValueError:
        late_sim.append({
            'noise_level': noise,
            'first_stage': 0,
            'late': np.nan,
            'se': np.nan,
        })

late_df = pd.DataFrame(late_sim)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: LATE vs noise level
valid = late_df.dropna()
ax1.plot(valid['noise_level'], valid['late'], 'o-', color='#EE1515', lw=2, markersize=5)
ax1.fill_between(valid['noise_level'],
                 valid['late'] - 1.96*valid['se'],
                 valid['late'] + 1.96*valid['se'],
                 color='#EE1515', alpha=0.15)
ax1.axhline(0, color='gray', ls=':', lw=1)
ax1.set_xlabel('Noise Level (fraction of flipped Z)')
ax1.set_ylabel('LATE Estimate')
ax1.set_title('LATE vs Instrument Noise')

# Right: LATE vs first-stage coefficient
ax2.scatter(valid['first_stage'], valid['late'], c=valid['noise_level'],
            cmap='RdYlBu_r', s=60, edgecolors='white', lw=0.5)
ax2.set_xlabel('First-Stage Coefficient')
ax2.set_ylabel('LATE Estimate')
ax2.set_title('LATE vs First-Stage Strength')
ax2.axhline(0, color='gray', ls=':', lw=1)
cb = fig.colorbar(ax2.collections[0], ax=ax2, label='Noise level')

fig.tight_layout()
plt.show()

print("As the instrument weakens (higher noise), the LATE estimate becomes:")
print("  - More variable (wider CIs)")
print("  - Potentially biased toward the OLS estimate (weak instrument bias)")

---
## Soul Badge + Volcano Badge Earned!

In [ ]:
badge_earned("Soul Badge + Volcano Badge", 6)